# Hybrid Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local **hybrid agentic workflow** for English-to-Chinese cross-lingual dialogue summarization.

Instead of using a small language model for every step, this pipeline combines our fine-tuned BART summarization model with local small language models served through Ollama.

The proposed pipeline is:

```text
English Dialogue
→ Agent 1: Fine-tuned BART Summarization Agent
→ Agent 2: Local SLM Chinese Translation Agent
→ Agent 3: Local SLM Evaluation-Revision Agent
→ Final Chinese Summary

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages
# Run this only once if the packages are not installed.

!pip install -U requests pandas tqdm transformers torch huggingface_hub safetensors hf_transfer

  Using cached hf_transfer-0.1.9-cp38-abi3-macosx_11_0_arm64.whl.metadata (1.7 kB)
Using cached hf_transfer-0.1.9-cp38-abi3-macosx_11_0_arm64.whl (1.3 MB)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [3]:
# Cell 2: Imports and global configuration

import os
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

# Enable fallback if some PyTorch operations are not fully supported on Apple Silicon MPS.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

OLLAMA_HOST = "http://localhost:11434"

# Agent 1: Fine-tuned BART summarization model
BART_MODEL_NAME = "yunu919/bart-large-dialogue-summarization"

# Local fine-tuned BART directory
LOCAL_BART_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/models/bart_large_dialogue_summarization_final"
)

# Agent 2 and Agent 3: Ollama local models
TRANSLATION_MODEL = "translategemma:12b"
EVALUATION_REVISION_MODEL = "qwen3.5:9b"

# If translategemma:12b is too slow, use:
# TRANSLATION_MODEL = "translategemma:4b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# BART generation settings
BART_MAX_INPUT_LENGTH = 768
BART_MAX_TARGET_LENGTH = 64
BART_NUM_BEAMS = 5

# Input dataset path
RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
FULL_OUTPUT_PATH = OUTPUT_DIR / "hybrid_bart_agent_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "hybrid_bart_agent_final_summaries.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "hybrid_bart_agent_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)
print("BART model:", BART_MODEL_NAME)
print("Translation model:", TRANSLATION_MODEL)
print("Evaluation-revision model:", EVALUATION_REVISION_MODEL)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_outputs.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_errors.jsonl
BART model: yunu919/bart-large-dialogue-summarization
Translation model: translategemma:12b
Evaluation-revision model: qwen3.5:9b


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

In [6]:
# Cell 5: Load fine-tuned BART summarization model from local directory

def get_torch_device() -> str:
    """Select the best available PyTorch device."""
    if torch.backends.mps.is_available():
        return "mps"

    if torch.cuda.is_available():
        return "cuda"

    return "cpu"


BART_DEVICE = get_torch_device()

print("Loading local fine-tuned BART model...")
print("Device:", BART_DEVICE)
print("Model path:", LOCAL_BART_DIR)

if not LOCAL_BART_DIR.exists():
    raise FileNotFoundError(f"Local BART directory not found: {LOCAL_BART_DIR}")

required_files = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

missing_files = [
    file_name for file_name in required_files
    if not (LOCAL_BART_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

print("Files in local BART directory:")
for file in sorted(LOCAL_BART_DIR.iterdir()):
    print("-", file.name)

bart_tokenizer = AutoTokenizer.from_pretrained(
    str(LOCAL_BART_DIR),
    local_files_only=True,
)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(LOCAL_BART_DIR),
    local_files_only=True,
    use_safetensors=True,
)

bart_model = bart_model.to(BART_DEVICE)
bart_model.eval()

print("Local fine-tuned BART model loaded successfully.")

Loading local fine-tuned BART model...
Device: mps
Model path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/models/bart_large_dialogue_summarization_final
Files in local BART directory:
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Local fine-tuned BART model loaded successfully.


## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [7]:
# Cell 6: Prompt templates for Agent 2 and Agent 3

TRANSLATION_PROMPT = """You are an expert English-to-Simplified-Chinese translator.
Your task is to translate the English summary accurately and naturally.

Guidelines:
1. Accuracy: Preserve all original meanings, names, numbers, dates, and entities without adding or removing information.
2. Localization: Use standard Chinese transliterations for English names. Ensure the phrasing is fluent and natural in Simplified Chinese.

English summary:
{english_summary}

Chinese summary:
"""


EVALUATION_REVISION_PROMPT = """You are an expert Chinese summary reviewer.
Your task is to evaluate and correct the Initial Chinese Summary based on the Original English Dialogue.

Review Criteria:
1. Factuality (Hallucination Check): Ensure the Chinese summary does not contain events, relationships, or temporal states that contradict the original dialogue.
2. Final Outcome Validation: Verify that the summary reflects the final agreement, accurately distinguishing it from jokes or canceled plans.
3. Naturalness: Ensure the Chinese text is fluent and concise, removing any unnecessary side details.

Instruction: If the Initial Chinese Summary has errors based on the criteria above, revise it. If it is accurate, output it as is. Do not provide explanations. Output ONLY the final revised Simplified Chinese summary.

Original English Dialogue:
{dialogue}

Intermediate English Summary:
{english_summary}

Initial Chinese Summary:
{chinese_summary}

Final revised Chinese summary:
"""

In [8]:
# Cell 7: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [9]:
# Cell 8: Agent functions

def bart_summarize(dialogue: str) -> str:
    """Generate an English summary using the fine-tuned BART model."""

    inputs = bart_tokenizer(
        dialogue,
        return_tensors="pt",
        truncation=True,
        max_length=BART_MAX_INPUT_LENGTH,
    )

    inputs = {key: value.to(BART_DEVICE) for key, value in inputs.items()}

    with torch.no_grad():
        summary_ids = bart_model.generate(
            **inputs,
            max_length=BART_MAX_TARGET_LENGTH,
            num_beams=BART_NUM_BEAMS,
            early_stopping=True,
        )

    summary = bart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return summary.strip()


def summarization_agent(dialogue: str) -> str:
    """Agent 1: Generate an intermediate English summary using fine-tuned BART."""
    return bart_summarize(dialogue)


def translation_agent(english_summary: str) -> str:
    """Agent 2: Translate the English summary into Simplified Chinese using Ollama."""
    prompt = TRANSLATION_PROMPT.format(english_summary=english_summary)

    return call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )


def evaluation_revision_agent(
    dialogue: str,
    english_summary: str,
    chinese_summary: str,
) -> str:
    """Agent 3: Evaluate and revise the Chinese summary using Ollama."""
    prompt = EVALUATION_REVISION_PROMPT.format(
        dialogue=dialogue,
        english_summary=english_summary,
        chinese_summary=chinese_summary,
    )

    return call_ollama(
        model=EVALUATION_REVISION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [10]:
# Cell 9: Hybrid BART-agent pipeline

def run_hybrid_bart_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the full hybrid pipeline on one example."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: Fine-tuned BART English summarization
    english_summary = summarization_agent(dialogue)

    # Agent 2: Chinese translation
    initial_chinese_summary = translation_agent(english_summary)

    # Agent 3: Evaluation and revision
    final_chinese_summary = evaluation_revision_agent(
        dialogue=dialogue,
        english_summary=english_summary,
        chinese_summary=initial_chinese_summary,
    )

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "agent1_model": BART_MODEL_NAME,
        "agent2_model": TRANSLATION_MODEL,
        "agent3_model": EVALUATION_REVISION_MODEL,
        "agent1_bart_english_summary": english_summary,
        "agent2_initial_chinese_summary": initial_chinese_summary,
        "agent3_final_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [11]:
# Cell 10: Load top examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [12]:
# Cell 11: Run the hybrid BART-agent pipeline for the first example

result = run_hybrid_bart_agent_pipeline(test_data[0])
result

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_model': 'yunu919/bart-large-dialogue-summarization',
 'agent2_model': 'translategemma:12b',
 'agent3_model': 'qwen3.5:9b',
 'agent1_bart_english_summary': "Hannah is looking for Betty's number. Amanda suggests Hannah to ask Larry, who called Betty last time they were at the park.",
 'agent2_initial_chinese_summary': '汉娜正在寻找贝蒂的电话号码。阿曼达建议汉娜去问拉里，因为上次他们在公园的

In [13]:
# Cell 12: Print hybrid BART-agent pipeline result clearly

def print_hybrid_bart_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: BART English Summary ===")
    print(result["agent1_bart_english_summary"])
    print()

    print("=== Agent 2: Initial Chinese Summary ===")
    print(result["agent2_initial_chinese_summary"])
    print()

    print("=== Agent 3: Final Revised Chinese Summary ===")
    print(result["agent3_final_chinese_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])


print_hybrid_bart_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: BART English Summary ===
Hannah is looking for Betty's number. Amanda suggests Hannah to ask Larry, who called Betty last time they were at the park.

=== Agent 2: Initial Chinese Summary ===
汉娜正在寻找贝蒂的电话号码。阿曼达建议汉娜去问拉里，因为上次他们在公园的时候，拉里给贝蒂打电话。

=== Agent 3: Final Revised Chinese Summary ===
汉娜在找贝蒂的电话号码，阿曼达建议她找拉里，因为上次在公园时拉里曾给贝蒂打过电话。

=== Reference English Summary ===
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

=== Reference Chinese Summary ===
汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。

=== Final Chinese Summary ===
汉娜在找贝蒂的电话

## 4. Save Results

This saves all intermediate outputs and the final output.


In [14]:
# Cell 13: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_outputs.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [15]:
# Cell 14: Batch inference with hybrid BART-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running hybrid BART-agent pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_hybrid_bart_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running hybrid BART-agent pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_outputs.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [16]:
# Cell 15: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "agent1_model": record.get("agent1_model", ""),
        "agent2_model": record.get("agent2_model", ""),
        "agent3_model": record.get("agent3_model", ""),
        "agent1_bart_english_summary": record.get("agent1_bart_english_summary", ""),
        "agent2_initial_chinese_summary": record.get("agent2_initial_chinese_summary", ""),
        "agent3_final_chinese_summary": record.get("agent3_final_chinese_summary", ""),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/hybrid_bart_agent_final_summaries.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_model,agent2_model,agent3_model,agent1_bart_english_summary,agent2_initial_chinese_summary,agent3_final_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",汉娜在找贝蒂的电话号码，阿曼达建议她找拉里，因为上次在公园时拉里曾给贝蒂打过电话。,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,yunu919/bart-large-dialogue-summarization,translategemma:12b,qwen3.5:9b,Hannah is looking for Betty's number. Amanda s...,汉娜正在寻找贝蒂的电话号码。阿曼达建议汉娜去问拉里，因为上次他们在公园的时候，拉里给贝蒂打电话。,汉娜在找贝蒂的电话号码，阿曼达建议她找拉里，因为上次在公园时拉里曾给贝蒂打过电话。
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,埃里克和罗布正在观看一位俄罗斯喜剧演员的脱口秀表演。,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,yunu919/bart-large-dialogue-summarization,translategemma:12b,qwen3.5:9b,Eric and Rob are watching a Russian comedian's...,埃里克和罗布正在观看一位俄罗斯喜剧演员的脱口秀表演。,埃里克和罗布正在观看一位俄罗斯喜剧演员的脱口秀表演。
2,test_00003,"Lenny: Babe, can you help me with something?\r...",莱尼将购买第一件或第三件紫色裤子。,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,yunu919/bart-large-dialogue-summarization,translategemma:12b,qwen3.5:9b,Lenny will buy the first or the third pair of ...,莱尼会从鲍勃那里购买第一件或第三件紫色的裤子。,莱尼将购买第一件或第三件紫色裤子。
3,test_00004,"Will: hey babe, what do you want for dinner to...",艾玛今晚不想做晚饭，她很快就会回家，威廉不会去接她。,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,yunu919/bart-large-dialogue-summarization,translategemma:12b,qwen3.5:9b,Emma doesn't want to cook dinner tonight. She ...,艾玛今晚不想做晚饭。她很快就会回家。威廉会去接她。,艾玛今晚不想做晚饭，她很快就会回家，威廉不会去接她。
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",简刚从摩洛哥回来。奥利和简将于周五晚上 6 点，在简的课程结束后见面喝茶。,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,yunu919/bart-large-dialogue-summarization,translategemma:12b,qwen3.5:9b,Jane is back from Morocco. Ollie and Jane will...,简从摩洛哥回到了。奥利和简将于周五晚上6点见面共进午餐，时间是简完成课程之后。,简刚从摩洛哥回来。奥利和简将于周五晚上 6 点，在简的课程结束后见面喝茶。


In [17]:
# Cell 16: Compare Agent 2 initial summary and Agent 3 final summary

comparison_columns = [
    "id",
    "agent1_bart_english_summary",
    "agent2_initial_chinese_summary",
    "agent3_final_chinese_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,agent1_bart_english_summary,agent2_initial_chinese_summary,agent3_final_chinese_summary,reference_chinese_summary
0,test_00001,Hannah is looking for Betty's number. Amanda s...,汉娜正在寻找贝蒂的电话号码。阿曼达建议汉娜去问拉里，因为上次他们在公园的时候，拉里给贝蒂打电话。,汉娜在找贝蒂的电话号码，阿曼达建议她找拉里，因为上次在公园时拉里曾给贝蒂打过电话。,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,test_00002,Eric and Rob are watching a Russian comedian's...,埃里克和罗布正在观看一位俄罗斯喜剧演员的脱口秀表演。,埃里克和罗布正在观看一位俄罗斯喜剧演员的脱口秀表演。,埃里克和罗伯要在youtube上看一场单口相声。
2,test_00003,Lenny will buy the first or the third pair of ...,莱尼会从鲍勃那里购买第一件或第三件紫色的裤子。,莱尼将购买第一件或第三件紫色裤子。,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,test_00004,Emma doesn't want to cook dinner tonight. She ...,艾玛今晚不想做晚饭。她很快就会回家。威廉会去接她。,艾玛今晚不想做晚饭，她很快就会回家，威廉不会去接她。,艾玛很快就会回家，而且她会告诉威尔。
4,test_00005,Jane is back from Morocco. Ollie and Jane will...,简从摩洛哥回到了。奥利和简将于周五晚上6点见面共进午餐，时间是简完成课程之后。,简刚从摩洛哥回来。奥利和简将于周五晚上 6 点，在简的课程结束后见面喝茶。,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [18]:
# Cell 17: Check whether Agent 3 changed the initial Chinese summary

if not df.empty:
    df["agent3_changed_output"] = (
        df["agent2_initial_chinese_summary"].str.strip()
        != df["agent3_final_chinese_summary"].str.strip()
    )

    change_summary = df["agent3_changed_output"].value_counts()

    print("Did Agent 3 change the output?")
    print(change_summary)

    display_columns = [
        "id",
        "agent3_changed_output",
        "agent2_initial_chinese_summary",
        "agent3_final_chinese_summary",
    ]

    df[display_columns]
else:
    print("No results found.")

Did Agent 3 change the output?
agent3_changed_output
True     4
False    1
Name: count, dtype: int64
